# Synthetic 3D Self-Training Demo

This notebook mirrors the lightweight unit tests in `tests/test_synthetic_pretraining.py`.

It uses tiny pure-Python toy models, not the full MONAI/PyTorch training stack, so it can run quickly anywhere. The goal is to visualize whether each self-training objective improves on synthetic 3D sphere volumes:

- Reconstruction
- Contrastive learning
- Diffusion-style denoising
- GAN-style adversarial reconstruction

In [ ]:
from tests.test_synthetic_pretraining import (
    TinyContrastiveEncoder,
    TinyDenoisingPretrainer,
    TinyDiscriminator,
    contrastive_pairs,
    diffusion_dataset,
    reconstruction_dataset,
)

SIZE = 8

## Helper Functions

In [ ]:
def pct_improvement(initial, final):
    return 100.0 * (initial - final) / initial


def run_reconstruction():
    dataset = reconstruction_dataset(size=SIZE, samples=12)
    model = TinyDenoisingPretrainer()
    initial = model.loss(dataset, SIZE)
    for _ in range(40):
        model.train_epoch(dataset, SIZE)
    final = model.loss(dataset, SIZE)
    return {"method": "reconstruction", "initial": initial, "final": final, "improvement": pct_improvement(initial, final)}


def run_contrastive():
    pairs = contrastive_pairs(size=SIZE)
    encoder = TinyContrastiveEncoder()
    initial = encoder.loss(pairs, SIZE)
    for _ in range(60):
        encoder.train_epoch(pairs, SIZE)
    final = encoder.loss(pairs, SIZE)
    return {"method": "contrastive", "initial": initial, "final": final, "improvement": pct_improvement(initial, final)}


def run_diffusion():
    dataset = diffusion_dataset(size=SIZE, samples=12)
    model = TinyDenoisingPretrainer()
    initial = model.loss(dataset, SIZE)
    for _ in range(45):
        model.train_epoch(dataset, SIZE, lr=0.07)
    final = model.loss(dataset, SIZE)
    return {"method": "diffusion", "initial": initial, "final": final, "improvement": pct_improvement(initial, final)}


def run_gan():
    dataset = reconstruction_dataset(size=SIZE, samples=10, seed=21)
    generator = TinyDenoisingPretrainer()
    discriminator = TinyDiscriminator()
    clean_volumes = [clean for _noisy, clean in dataset]

    initial = generator.loss(dataset, SIZE)
    for _ in range(30):
        generator.train_epoch(dataset, SIZE, lr=0.06)
        generated = [generator.reconstruct(noisy, SIZE) for noisy, _clean in dataset]
        discriminator.train_epoch(clean_volumes, generated, SIZE)

    final = generator.loss(dataset, SIZE)
    generated = [generator.reconstruct(noisy, SIZE) for noisy, _clean in dataset]
    disc_acc = discriminator.accuracy(clean_volumes, generated, SIZE)
    return {
        "method": "gan",
        "initial": initial,
        "final": final,
        "improvement": pct_improvement(initial, final),
        "discriminator_accuracy": disc_acc,
    }

## Run All Four Synthetic Objectives

In [ ]:
results = [run_reconstruction(), run_contrastive(), run_diffusion(), run_gan()]

header = f"{'method':<16} {'initial':>12} {'final':>12} {'improvement':>14} {'disc_acc':>10}"
print(header)
print('-' * len(header))
for row in results:
    disc = row.get('discriminator_accuracy')
    disc_text = f"{disc:.2%}" if disc is not None else '-'
    print(f"{row['method']:<16} {row['initial']:>12.6f} {row['final']:>12.6f} {row['improvement']:>13.2f}% {disc_text:>10}")

Expected reference output from the current test run:

```text
reconstruction: initial_loss=0.043076 final_loss=0.020939 improvement=51.39%
contrastive: initial_loss=0.005044 final_loss=0.000495 improvement=90.18%
diffusion: initial_loss=0.043263 final_loss=0.022034 improvement=49.07%
gan: initial_reconstruction=0.042939 final_reconstruction=0.027283 reconstruction_improvement=36.46% discriminator_accuracy=100.00%
```